----
# SLH-DSA
----

In [32]:
import os

# Use the locally built liboqs install if available
oqs_install_path = r"C:\Users\Admin\_oqs_0.15.0"
if os.path.isdir(oqs_install_path):
    os.environ["OQS_INSTALL_PATH"] = oqs_install_path
    print("Using OQS_INSTALL_PATH:", os.environ["OQS_INSTALL_PATH"])

import oqs
import time
import numpy as np
import matplotlib.pyplot as plt
import csv

benchmark_output_path = "slh_dsa_benchmark_results.txt"
csv_output_path = "slh_dsa_timing.csv"
with open(benchmark_output_path, "w", encoding="utf-8") as f:
    f.write("SLH-DSA Benchmark Results\n")
    f.write("=" * 40 + "\n")

csv_enabled = True
try:
    with open(csv_output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(["metric", "run", "time_ms"])
except PermissionError:
    print(f"Warning: Cannot write to {csv_output_path} (file may be open in another program). CSV logging disabled.")
    csv_enabled = False




def append_to_output(*lines):
    with open(benchmark_output_path, "a", encoding="utf-8") as f:
        for line in lines:
            f.write(str(line) + "\n")


def append_to_csv(metric, run, time_ms):
    if csv_enabled:
        with open(csv_output_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f, delimiter=';')
            writer.writerow([metric, run, time_ms])



Using OQS_INSTALL_PATH: C:\Users\Admin\_oqs_0.15.0


In [33]:
# Test SLH-DSA availability
print("Testing SLH-DSA availability...")
print("\nAvailable SLH-DSA algorithms:")
slh_algs = [alg for alg in oqs.get_enabled_sig_mechanisms() if "SLH" in alg.upper()]
for alg in slh_algs:
    print(f"  ✓ {alg}")

if not slh_algs:
    print("No SLH-DSA algorithms enabled in this OQS build.")
else:
    print("\n✓ SLH-DSA setup successful!")

Testing SLH-DSA availability...

Available SLH-DSA algorithms:
  ✓ SLH_DSA_PURE_SHA2_128S
  ✓ SLH_DSA_PURE_SHA2_128F
  ✓ SLH_DSA_PURE_SHA2_192S
  ✓ SLH_DSA_PURE_SHA2_192F
  ✓ SLH_DSA_PURE_SHA2_256S
  ✓ SLH_DSA_PURE_SHA2_256F
  ✓ SLH_DSA_PURE_SHAKE_128S
  ✓ SLH_DSA_PURE_SHAKE_128F
  ✓ SLH_DSA_PURE_SHAKE_192S
  ✓ SLH_DSA_PURE_SHAKE_192F
  ✓ SLH_DSA_PURE_SHAKE_256S
  ✓ SLH_DSA_PURE_SHAKE_256F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_128S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_128F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_192S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_192F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_256S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_256F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_128S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_128F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_192S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_192F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_256S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_256F
  ✓ SLH_DSA_SHA2_256_PREHASH_SHA2_128S
  ✓ SLH_DSA_SHA2_256_PREHASH_SHA2_128F
  ✓ SLH_DSA_SHA2_256_PREHASH_SHA2_192S
  ✓ SLH_DSA_SHA2

In [34]:
# Benchmark SLH-DSA keypair generation for a representative parameter set
alg = "SLH_DSA_PURE_SHA2_256F"
if alg not in slh_algs:
    raise ValueError(f"Algorithm {alg} is not enabled. Choose one of: {slh_algs}")

print(f"Benchmarking {alg} keypair generation...")
runs = 20
times_ms = []
with oqs.Signature(alg) as sig:
    for run_index in range(runs):
        start = time.perf_counter()
        _ = sig.generate_keypair()
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        times_ms.append(elapsed_ms)
        append_to_csv("keypair_generation", run_index + 1, elapsed_ms)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean keypair time: {np.mean(times_ms):.3f} ms")
print(f"Min keypair time: {np.min(times_ms):.3f} ms")
print(f"Max keypair time: {np.max(times_ms):.3f} ms")
print("Times (ms):", [round(t, 3) for t in times_ms])

append_to_output(
    "\nBenchmarking {alg} keypair generation...".format(alg=alg),
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean keypair time: {np.mean(times_ms):.3f} ms",
    f"Min keypair time: {np.min(times_ms):.3f} ms",
    f"Max keypair time: {np.max(times_ms):.3f} ms",

    f"Times (ms): {[round(t, 3) for t in times_ms]}")

Benchmarking SLH_DSA_PURE_SHA2_256F keypair generation...
Algorithm: SLH_DSA_PURE_SHA2_256F
Runs: 20
Mean keypair time: 24.850 ms
Min keypair time: 21.335 ms
Max keypair time: 29.137 ms
Times (ms): [29.137, 28.817, 24.587, 21.335, 22.113, 23.696, 22.498, 25.658, 26.547, 23.399, 27.882, 22.141, 24.486, 25.243, 21.85, 23.457, 26.272, 26.995, 25.01, 25.877]


In [35]:
# Benchmark SLH-DSA signing time
message = b"Hello, world! This is a test message for SLH-DSA signing benchmark."

print(f"Benchmarking {alg} signing...")
runs = 20
sign_times_ms = []
with oqs.Signature(alg) as sig:
    pk = sig.generate_keypair()
    context = b""
    for run_index in range(runs):
        start = time.perf_counter()
        signature = sig.sign_with_ctx_str(message, context)
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        sign_times_ms.append(elapsed_ms)
        append_to_csv("signing", run_index + 1, elapsed_ms)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean signing time: {np.mean(sign_times_ms):.3f} ms")
print(f"Min signing time: {np.min(sign_times_ms):.3f} ms")
print(f"Max signing time: {np.max(sign_times_ms):.3f} ms")
print("Signing times (ms):", [round(t, 3) for t in sign_times_ms])

append_to_output(
    "\nBenchmarking {alg} signing...".format(alg=alg),
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean signing time: {np.mean(sign_times_ms):.3f} ms",
    f"Min signing time: {np.min(sign_times_ms):.3f} ms",
    f"Max signing time: {np.max(sign_times_ms):.3f} ms",

    f"Signing times (ms): {[round(t, 3) for t in sign_times_ms]}")

Benchmarking SLH_DSA_PURE_SHA2_256F signing...
Algorithm: SLH_DSA_PURE_SHA2_256F
Runs: 20
Mean signing time: 471.568 ms
Min signing time: 407.100 ms
Max signing time: 571.005 ms
Signing times (ms): [465.723, 489.981, 468.448, 407.1, 479.782, 467.758, 426.08, 435.277, 485.582, 571.005, 479.375, 483.271, 481.068, 489.603, 447.691, 453.244, 466.683, 508.72, 438.412, 486.561]


In [36]:
# SLH-DSA Verification Benchmark
# Note: Verification is currently not working due to a possible bug in the oqs library for SLH-DSA algorithms.
# The regular verify and verify_with_ctx_str methods return False even with correct signatures.
# This may be due to the library version or implementation issues.
# Skipping verification benchmark for now.

In [37]:
# Measure SLH-DSA size metrics
print(f"Measuring {alg} size metrics...")
with oqs.Signature(alg) as sig:
    pk = sig.generate_keypair()
    sk = sig.export_secret_key()
    context = b""
    signature = sig.sign_with_ctx_str(message, context)

print(f"Algorithm: {alg}")
print(f"Public key size: {len(pk)} bytes")
print(f"Secret key size: {len(sk)} bytes")
print(f"Signature size: {len(signature)} bytes")

append_to_output(
    "\nMeasuring {alg} size metrics...".format(alg=alg),
    f"Algorithm: {alg}",
    f"Public key size: {len(pk)} bytes",
    f"Secret key size: {len(sk)} bytes",

    f"Signature size: {len(signature)} bytes")

Measuring SLH_DSA_PURE_SHA2_256F size metrics...
Algorithm: SLH_DSA_PURE_SHA2_256F
Public key size: 64 bytes
Secret key size: 128 bytes
Signature size: 49856 bytes
